In [ ]:
# Homework 12: https://almondine-song-c43.notion.site/Homework-12-1cc34d22fd9a430bb2b210cb9ce3db71

In [19]:
# PROVER
import galois
import numpy as np
import random
from functools import reduce
from py_ecc.bn128 import G1, G2, add, multiply, curve_order, field_modulus, pairing, neg

GF = galois.GF(curve_order, primitive_element=5, verify=False)

# langrange interpolation
def interpolate_column(col):
    xs = GF(np.arange(1, len(col) + 1)) # x-vector
    return galois.lagrange_poly(xs, col) # interpolating y-vector through points in col

# innter product over multiple polynomials (multiply each polynomial with witness coefficient)
def inner_product_polynomials_with_witness(polys, witness):
    mul_ = lambda x, y: x * y
    sum_ = lambda x, y: x + y
    return reduce(sum_, map(mul_, polys, witness))

# generate structured reference string (srs) over G
def generate_srs(discrete_log, G, n):
    powers = [pow(discrete_log, i, curve_order) for i in range(n)]
    return [multiply(G, power) for power in powers]

# multiply polynomial coeffs with srs
def mul_poly_with_srs(poly_coeffs, point_vector):
    mul_ = lambda x, y: multiply(x, y)
    sum_ = lambda x, y: add(x, y)
    return reduce(sum_, map(mul_, point_vector, poly_coeffs))

def eval_polynomials(polys, discrete_log):
    return np.array([poly(discrete_log) for poly in polys])

def generate_proof(R1CS, a):
    A = R1CS[0]
    B = R1CS[1]
    C = R1CS[2]

    L_galois = GF((A + curve_order) % curve_order)
    R_galois = GF((B + curve_order) % curve_order)
    O_galois = GF((C + curve_order) % curve_order)
    a_galois = GF(np.array([1, out, x, y, v1, v2]))
    
    # check conversion
    assert all(
        np.equal(
            np.matmul(L_galois, a_galois) * np.matmul(R_galois, a_galois),
            np.matmul(O_galois, a_galois),
        )
    ), "not equal"

    # Lagrange interpolation
    U_polys = np.apply_along_axis(interpolate_column, 0, L_galois)
    V_polys = np.apply_along_axis(interpolate_column, 0, R_galois)
    W_polys = np.apply_along_axis(interpolate_column, 0, O_galois)

    # Inner product with witness
    Ua = inner_product_polynomials_with_witness(U_polys, a_galois)
    Va = inner_product_polynomials_with_witness(V_polys, a_galois)
    Wa = inner_product_polynomials_with_witness(W_polys, a_galois)

    # Calculate t
    t = (
        galois.Poly([1, curve_order-1], field=GF)
        * galois.Poly([1, curve_order-2], field=GF)
        * galois.Poly([1, curve_order-3], field=GF)
    )
    # print("t(x): ", t)

    # calculate h
    h = (Ua * Va - Wa) // t
    
    # print("h(x): ", h)
    assert Ua * Va == Wa + h * t, "division has a remainder"

    # Structured reference string (SRS)
    tau = random.randint(1,1000) % curve_order
    alpha = random.randint(1,1000) % curve_order
    beta = random.randint(1,1000) % curve_order
    gamma =  random.randint(1,1000) % curve_order
    gamma_inv = pow(gamma, -1, curve_order) % curve_order
    delta =  random.randint(1,1000) % curve_order
    delta_inv = pow(delta, -1, curve_order) % curve_order

    r =  random.randint(1,1000) % curve_order
    s =  random.randint(1,1000) % curve_order
    
    srs_G1 = generate_srs(tau,G1,3)
    srs_G2 = generate_srs(tau,G2,3)
    
    t_tau = t(GF(tau)) * delta_inv
    srs_HT = [multiply(ec_point, int(t_tau)) for ec_point in generate_srs(tau,G1, 3)]

    Alpha1 = multiply(G1, alpha)
    Beta1 = multiply(G1, beta)
    Beta2 = multiply(G2, beta)
    Gamma2 = multiply(G2, gamma)
    Delta1 = multiply(G1, delta)
    Delta2 = multiply(G2, delta)
    
    # generate proof
    poly_coeffs = [int(coeff) for coeff in Ua.coefficients(size=L_galois.shape[0],order="asc")]
    A1 = add(add(mul_poly_with_srs(poly_coeffs, srs_G1), Alpha1), multiply(Delta1, r))
    
    poly_coeffs = [int(coeff) for coeff in Va.coefficients(size=R_galois.shape[0],order="asc")]
    B1 = add(add(mul_poly_with_srs(poly_coeffs, srs_G1), Beta1), multiply(Delta1, s))
    B2 = add(add(mul_poly_with_srs(poly_coeffs, srs_G2), Beta2), multiply(Delta2, s))
    
    poly_coeffs = [int(coeff) for coeff in h.coefficients(size=O_galois.shape[0],order="asc")]
    HT = mul_poly_with_srs(poly_coeffs, srs_HT) # divide by delta
    print(HT)

    Eps = alpha * eval_polynomials(V_polys, tau) + beta * eval_polynomials(U_polys,tau) + eval_polynomials(W_polys,tau)
    # Eps_a = inner_product_polynomials_with_witness(Eps, a_galois)
    # Eps1 = multiply(G1, int(Eps_a))
    # C1 = add(Eps1,HT)
    
    l = 3
    a_pub = a_galois[:l]
    a_priv = a_galois[l:]

    Eps_pub = Eps[:l]
    Eps_priv = Eps[l:]
    
    Eps_pub_a = inner_product_polynomials_with_witness(Eps_pub, a_pub) * gamma_inv 
    Eps_priv_a = inner_product_polynomials_with_witness(Eps_priv, a_priv) * delta_inv

    Eps_pub1 = multiply(G1, int(Eps_pub_a))
    Eps_priv1 = multiply(G1, int(Eps_priv_a))

    X1 = Eps_pub1
    C1 = add(add(add(add(Eps_priv1,HT),multiply(A1,s)),multiply(B1,r)),multiply(Delta1,(curve_order - r*s % curve_order) % curve_order ))

    return (A1, B2, C1, X1, Alpha1, Beta2, Gamma2, Delta2)

In [20]:
# Define the matrices
A = np.array([[0,0,3,0,0,0],
               [0,0,0,0,1,0],
               [0,0,1,0,0,0]])

B = np.array([[0,0,1,0,0,0],
               [0,0,0,1,0,0],
               [0,0,0,5,0,0]])

C = np.array([[0,0,0,0,1,0],
               [0,0,0,0,0,1],
               [-3,1,1,2,0,-1]])

# L_galois = GF((A + curve_order) % curve_order)
# R_galois = GF((B + curve_order) % curve_order)
# O_galois = GF((C + curve_order) % curve_order)

# pick values for x and y
x = GF(random.randint(1,1000))
y = GF(random.randint(1,1000))

# this is our orignal formula: 3 * x * x * y + 5 * x * y + (-1 * x) + (-2 * y) + 3

# result given chosen witness in GF
out = GF(3) * x * x * y + GF(5) * x * y + GF(-1 + curve_order)* x + GF(-2 + curve_order) * y + GF(3)

# intermediate values
v1 = GF(3) * x * x
v2 = v1 * y

# witness vector
a = np.array([1, out, x, y, v1, v2])

A1, B2, C1, X1, Alpha, Beta, Gamma, Delta = generate_proof([A,B,C], a)


(3536626470012464894439673914737798775053780901558327358560775037731236893312, 4509164599163663490230431243762516257786216792821821667951653607659194201540)


In [21]:
# VERIFIER
from py_ecc.bn128.bn128_curve import FQ12
from py_ecc.bn128 import eq, neg, pairing, final_exponentiate

eq(FQ12.one(), final_exponentiate(pairing(B2, neg(A1)) * pairing(Beta, Alpha) * pairing(Gamma, X1) * pairing(Delta, C1)))

True